# Two-Stage Book Recommender: SVD Retrieval + LightGBM Ranker (corrected)

**What changed and why.** The old ranker used `Is_Explicit` as a feature while the label was `Book-Rating > 0` (the same thing), and its rating-based features were computed from all ratings, so test information leaked in. It was also trained on explicit-vs-implicit rows but served SVD candidates, a different distribution. This version splits explicit ratings into train / valid / test, recomputes every feature from train only, retrieves 200 SVD candidates per user, labels them with held-out positives (rating >= 7), and trains a `lambdarank` model with one group per user. The valid split supplies ranker labels so the test split is only touched by evaluation. Evaluation now uses relevant = rating >= 7, users with >= 5 train ratings, a correct NDCG (ideal DCG from `min(#relevant, K)`), MAP, retrieval Recall@200, and popularity and SVD-only baselines, with precomputed lookups instead of per-user DataFrame scans.

Inputs: `interaction_features.csv` (only `User-ID`, `ISBN`, `Book-Rating` are read) and `books_cleaned_engineered.csv`. `user_features.csv` and `book_features.csv` are no longer used because they were built from all ratings.

## 0. Config

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "."                 # use "/content" in Colab
SEED = 42
REL_THRESHOLD = 7              # rating >= 7 is relevant
MIN_TRAIN_RATINGS = 5          # eligibility for ranker training and evaluation
K = 10
N_CANDIDATES = 200
N_FACTORS = 64
VALID_FRAC, TEST_FRAC = 0.15, 0.20
RANKER_MAX_USERS = None        # cap on users used to train the ranker (None = all eligible)
EVAL_MAX_USERS = None          # None = all eligible test users, int = seeded random sample
BATCH = 128                    # users scored per matrix multiply (lower it if memory is tight)

rng = np.random.default_rng(SEED)

## 1. Data split

In [ ]:
inter = pd.read_csv(f"{DATA_DIR}/interaction_features.csv",
                    usecols=["User-ID", "ISBN", "Book-Rating"])
books = (pd.read_csv(f"{DATA_DIR}/books_cleaned_engineered.csv",
                     usecols=["ISBN", "Book-Title", "Book-Author"])
           .drop_duplicates("ISBN"))

inter = inter.dropna(subset=["User-ID", "ISBN"]).copy()
inter["User-ID"] = inter["User-ID"].astype("int64")
inter["ISBN"] = inter["ISBN"].astype(str)
inter["Book-Rating"] = inter["Book-Rating"].astype("float32")
inter = inter.drop_duplicates(["User-ID", "ISBN"], keep="last")

explicit = inter[inter["Book-Rating"] > 0].reset_index(drop=True)
implicit = inter[inter["Book-Rating"] == 0].reset_index(drop=True)

# Three-way split: the ranker needs labels that are not the test labels, so
# "valid" supplies ranker labels and "test" is only touched by evaluation.
r = rng.random(len(explicit))
part = np.where(r < TEST_FRAC, "test",
                np.where(r < TEST_FRAC + VALID_FRAC, "valid", "train"))
train_df, valid_df, test_df = (explicit[part == p].copy() for p in ("train", "valid", "test"))

print(f"explicit train/valid/test: {len(train_df):,} / {len(valid_df):,} / {len(test_df):,}")
print(f"implicit interactions:     {len(implicit):,}")

## 2. Features (train split only)

In [ ]:
# Index spaces come from train only, so every retrievable item has train signal.
# Users = anyone with train-explicit or implicit history (implicit-only users get the popularity fallback).
item_ids = np.sort(train_df["ISBN"].unique())
user_ids = np.union1d(train_df["User-ID"].unique(), implicit["User-ID"].unique())
item_map = pd.Series(np.arange(len(item_ids)), index=item_ids)
user_map = pd.Series(np.arange(len(user_ids)), index=user_ids)
n_items, n_users = len(item_ids), len(user_ids)

def encode(df):
    out = df.copy()
    out["u"] = out["User-ID"].map(user_map).fillna(-1).astype(np.int64)   # -1 = not in train index space
    out["i"] = out["ISBN"].map(item_map).fillna(-1).astype(np.int64)
    return out

train, valid, test, imp = encode(train_df), encode(valid_df), encode(test_df), encode(implicit)
print(f"Index space -> users: {n_users:,} | items: {n_items:,}")

global_mean = float(train["Book-Rating"].mean())
global_std = float(train["Book-Rating"].std())

# User-level table, one row per user index (cold users get global fallbacks)
user_feat = (train.groupby("u")["Book-Rating"]
                  .agg(User_Average_Rating="mean", User_Rating_Count="count", User_Rating_Std="std")
                  .reindex(np.arange(n_users)))
user_feat["User_Average_Rating"] = user_feat["User_Average_Rating"].fillna(global_mean)
user_feat["User_Rating_Count"] = user_feat["User_Rating_Count"].fillna(0)
user_feat["User_Rating_Std"] = user_feat["User_Rating_Std"].fillna(global_std)   # also covers count == 1
user_feat["User_Implicit_Count"] = imp.groupby("u").size().reindex(user_feat.index).fillna(0)

# Item-level table, one row per item index
item_feat = (train.groupby("i")["Book-Rating"]
                  .agg(Book_Average_Rating="mean", Book_Rating_Count="count")
                  .reindex(np.arange(n_items)))
item_feat["Book_Average_Rating"] = item_feat["Book_Average_Rating"].fillna(global_mean)
item_feat["Book_Rating_Count"] = item_feat["Book_Rating_Count"].fillna(0)
imp_item_cnt = imp[imp["i"] >= 0].groupby("i").size().reindex(item_feat.index).fillna(0)
item_feat["Log_Book_Popularity"] = np.log1p(item_feat["Book_Rating_Count"] + imp_item_cnt)

USER_COLS = ["User_Average_Rating", "User_Rating_Count", "User_Rating_Std", "User_Implicit_Count"]
ITEM_COLS = ["Book_Average_Rating", "Book_Rating_Count", "Log_Book_Popularity"]
FEATURES = USER_COLS + ITEM_COLS + ["svd_score", "svd_rank"]

user_arr = user_feat[USER_COLS].to_numpy(np.float32)
item_arr = item_feat[ITEM_COLS].to_numpy(np.float32)
train_cnt = user_feat["User_Rating_Count"].to_numpy()

def add_features(cand):
    """Row-aligned feature matrix: user features by user index, item features by item index. No merges."""
    mat = np.hstack([
        user_arr[cand["u"].to_numpy()],
        item_arr[cand["i"].to_numpy()],
        cand[["svd_score", "svd_rank"]].to_numpy(np.float32),
    ])
    return pd.DataFrame(mat, columns=FEATURES)

## 3. Retrieval (mean-centered TruncatedSVD)

In [ ]:
# Mean-center per user so the factors model deviations from each user's own baseline.
train["centered"] = train["Book-Rating"] - train["u"].map(user_feat["User_Average_Rating"])
R = csr_matrix((train["centered"].to_numpy(np.float32), (train["u"], train["i"])),
               shape=(n_users, n_items))

svd = TruncatedSVD(n_components=N_FACTORS, random_state=SEED)
user_factors = svd.fit_transform(R).astype(np.float32)
item_factors = svd.components_.T.astype(np.float32)

# Users with no explicit train ratings (or an all-zero factor vector) fall back to popularity.
has_signal = (train_cnt > 0) & (np.abs(user_factors).sum(axis=1) > 1e-9)
print(f"Users with SVD signal: {has_signal.sum():,} / {n_users:,}")

pop_order = (item_feat.sort_values(["Book_Rating_Count", "Book_Average_Rating"], ascending=False)
                      .index.to_numpy())

def build_seen(*frames):
    rows = np.concatenate([f.loc[(f["u"] >= 0) & (f["i"] >= 0), "u"].to_numpy() for f in frames])
    cols = np.concatenate([f.loc[(f["u"] >= 0) & (f["i"] >= 0), "i"].to_numpy() for f in frames])
    return csr_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(n_users, n_items))

seen_train = build_seen(train, imp)           # masks used while building ranker training data
seen_eval = build_seen(train, imp, valid)     # at test time valid ratings are known history too

def seen_items(seen, u):
    return seen.indices[seen.indptr[u]:seen.indptr[u + 1]]

def pop_unseen(seen_arr, k):
    head = pop_order[: k + len(seen_arr)]     # guarantees >= k unseen items when the catalogue is large enough
    return head[~np.isin(head, seen_arr)][:k]

def _topk(scores, k):
    n = scores.shape[1]
    k = min(k, n)                             # argpartition needs kth < n
    if k < n:
        idx = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    else:
        idx = np.tile(np.arange(n), (scores.shape[0], 1))
    part = np.take_along_axis(scores, idx, axis=1)
    order = np.argsort(-part, axis=1)
    return np.take_along_axis(idx, order, axis=1), np.take_along_axis(part, order, axis=1)

## 4. Candidate generation

In [ ]:
def retrieve(user_idx, seen, top_k=N_CANDIDATES, batch=BATCH):
    """Top-k unseen items per user by SVD dot product (popularity for users without signal).
    Returns one row per (user, candidate) with svd_score and svd_rank; rows are contiguous per user."""
    user_idx = np.asarray(user_idx)
    k = min(top_k, n_items)
    parts = []
    for s in range(0, len(user_idx), batch):
        b = user_idx[s:s + batch]
        scores = user_factors[b] @ item_factors.T                 # (batch, n_items)
        coo = seen[b].tocoo()
        scores[coo.row, coo.col] = -np.inf
        idx, sc = _topk(scores, k)
        for row in np.where(~has_signal[b])[0]:                   # popularity fallback
            lst = pop_unseen(seen_items(seen, b[row]), k)
            idx[row, :] = -1
            sc[row, :] = -np.inf
            idx[row, :len(lst)] = lst
            sc[row, :len(lst)] = 0.0
        ok = np.isfinite(sc).ravel()
        parts.append(pd.DataFrame({
            "u": np.repeat(b, k)[ok],
            "i": idx.ravel()[ok],
            "svd_score": sc.ravel()[ok],
            "svd_rank": np.tile(np.arange(1, k + 1), len(b))[ok],
        }))
    out = pd.concat(parts, ignore_index=True)
    return out.astype({"u": "int64", "i": "int64", "svd_score": "float32", "svd_rank": "float32"})

def label_candidates(cand, pos):
    pos = pos[pos["i"] >= 0]
    pos_keys = pos["u"].to_numpy(np.int64) * n_items + pos["i"].to_numpy(np.int64)
    keys = cand["u"].to_numpy(np.int64) * n_items + cand["i"].to_numpy(np.int64)
    cand["label"] = np.isin(keys, pos_keys).astype(np.int8)
    return cand

## 5. Ranker training (LightGBM lambdarank, one group per user)

In [ ]:
eligible = np.where(train_cnt >= MIN_TRAIN_RATINGS)[0]
valid_pos = valid[(valid["Book-Rating"] >= REL_THRESHOLD) & (valid["u"] >= 0)]

ranker_users = np.intersect1d(valid_pos["u"].unique(), eligible)
if RANKER_MAX_USERS is not None and len(ranker_users) > RANKER_MAX_USERS:
    ranker_users = np.sort(rng.choice(ranker_users, RANKER_MAX_USERS, replace=False))

train_cand = label_candidates(retrieve(ranker_users, seen_train), valid_pos)

# Groups with no positive carry no lambdarank signal, so drop them.
has_pos = train_cand.groupby("u")["label"].transform("max") > 0
n_total_pos = int(valid_pos[valid_pos["u"].isin(ranker_users)].shape[0])
print(f"Ranker users: {len(ranker_users):,} | with >=1 positive among {N_CANDIDATES} candidates: "
      f"{train_cand.loc[has_pos, 'u'].nunique():,}")
print(f"Positives retrieved: {int(train_cand['label'].sum()):,} of {n_total_pos:,} valid positives "
      f"(retrieval recall@{N_CANDIDATES} on valid ~ {train_cand['label'].sum() / max(n_total_pos, 1):.3f})")
train_cand = train_cand[has_pos]
if train_cand["u"].nunique() < 50:
    raise ValueError("Too few ranker groups with positives. Raise VALID_FRAC or N_CANDIDATES, or lower MIN_TRAIN_RATINGS.")

# User-level validation split: no user appears in both train and validation groups.
users_with_pos = train_cand["u"].unique()
rng.shuffle(users_with_pos)
n_val = max(1, int(0.15 * len(users_with_pos)))
val_users = set(users_with_pos[:n_val].tolist())
is_val = train_cand["u"].isin(val_users).to_numpy()

def make_dataset(df, reference=None):
    df = df.sort_values(["u", "svd_rank"], kind="stable")
    groups = df.groupby("u", sort=True).size().to_numpy()
    return lgb.Dataset(add_features(df), label=df["label"].to_numpy(), group=groups, reference=reference)

dtrain = make_dataset(train_cand[~is_val])
dvalid = make_dataset(train_cand[is_val], reference=dtrain)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [K],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": SEED,
    "verbose": -1,
}
ranker = lgb.train(params, dtrain, num_boost_round=500, valid_sets=[dvalid],
                   callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(50)])
best_iter = ranker.best_iteration or None
print(f"Best iteration: {ranker.best_iteration}")
print(pd.Series(ranker.feature_importance("gain"), index=FEATURES).sort_values(ascending=False).round(1))

## 6. Inference (retrieve, rank, top-K)

In [ ]:
def recommend(user_id, k=K):
    """Top-k books for one user. Unknown users, or users without explicit train ratings, get popularity."""
    u = int(user_map.get(user_id, -1))
    if u < 0 or not has_signal[u]:
        top = pop_unseen(seen_items(seen_eval, u) if u >= 0 else np.array([], dtype=np.int64), k)
        out = pd.DataFrame({"ISBN": item_ids[top], "rank_score": np.nan})
    else:
        cand = retrieve([u], seen_eval)
        cand["rank_score"] = ranker.predict(add_features(cand), num_iteration=best_iter)
        top = cand.nlargest(k, "rank_score")
        out = pd.DataFrame({"ISBN": item_ids[top["i"].to_numpy()], "rank_score": top["rank_score"].to_numpy()})
    return out.merge(books, on="ISBN", how="left")[["ISBN", "Book-Title", "Book-Author", "rank_score"]]

## 7. Evaluation

In [ ]:
# ---- ground truth, precomputed once (no per-user DataFrame scans) ----
test_pos = test[(test["Book-Rating"] >= REL_THRESHOLD) & (test["u"] >= 0)]
n_rel = test_pos.groupby("u").size().to_dict()   # includes relevant items the retriever cannot reach (cold items)
rel_items = {u: g["i"].to_numpy() for u, g in test_pos[test_pos["i"] >= 0].groupby("u")}

eval_users = np.intersect1d(test_pos["u"].unique(), eligible)
if EVAL_MAX_USERS is not None and len(eval_users) > EVAL_MAX_USERS:
    eval_users = np.sort(rng.choice(eval_users, EVAL_MAX_USERS, replace=False))
print(f"Evaluating {len(eval_users):,} users (>= {MIN_TRAIN_RATINGS} train ratings and >= 1 relevant test item)")

EMPTY = np.empty(0, dtype=np.int64)

def user_metrics(recs, rel, nrel, k=K):
    recs = recs[:k]
    hits = np.isin(recs, rel).astype(np.float64)
    n_hit = hits.sum()
    disc = 1.0 / np.log2(np.arange(2, len(recs) + 2))
    ideal = (1.0 / np.log2(np.arange(2, min(nrel, k) + 2))).sum()
    ap = (hits * np.cumsum(hits) / np.arange(1, len(recs) + 1)).sum() / min(nrel, k)
    return n_hit / k, n_hit / nrel, (hits * disc).sum() / ideal, ap

def evaluate(rec_lists, k=K):
    rows = [user_metrics(rec_lists.get(u, EMPTY), rel_items.get(u, EMPTY), n_rel[u], k) for u in eval_users]
    p, rc, nd, ap = np.mean(rows, axis=0)
    return {f"Precision@{k}": p, f"Recall@{k}": rc, f"NDCG@{k}": nd, f"MAP@{k}": ap}

def recall_at_n(rec_lists):
    return float(np.mean([np.isin(rec_lists.get(u, EMPTY), rel_items.get(u, EMPTY)).sum() / n_rel[u]
                          for u in eval_users]))

def to_lists(df):
    return {u: g["i"].to_numpy() for u, g in df.groupby("u", sort=False)}

# ---- candidates for every eval user (seen = train + implicit + valid) ----
eval_cand = retrieve(eval_users, seen_eval)

# Baseline 1: global popularity
pop_lists = {u: pop_unseen(seen_items(seen_eval, u), N_CANDIDATES) for u in eval_users}

# Baseline 2: SVD only (candidates are already in SVD order)
svd_lists = to_lists(eval_cand)

# SVD + LightGBM: rerank the same candidates
eval_cand["rank_score"] = ranker.predict(add_features(eval_cand), num_iteration=best_iter)
rank_lists = to_lists(eval_cand.sort_values(["u", "rank_score"], ascending=[True, False], kind="stable"))

results = pd.DataFrame([
    {"Model": "Popularity", **evaluate(pop_lists)},
    {"Model": "SVD only", **evaluate(svd_lists)},
    {"Model": "SVD + LightGBM", **evaluate(rank_lists)},
]).set_index("Model")

print(f"\nRetrieval Recall@{N_CANDIDATES}  SVD: {recall_at_n(svd_lists):.4f} | Popularity: {recall_at_n(pop_lists):.4f}")
print(f"(The ranker only reorders the SVD candidates, so its ceiling is the SVD Recall@{N_CANDIDATES}.)\n")
print(results.round(4).to_string())
results.to_csv("two_stage_evaluation_results.csv")

## 8. Sample recommendation

In [ ]:
demo_user = int(user_ids[eval_users[0]])
print(f"Top-{K} for User-ID {demo_user}")
print(recommend(demo_user, K).to_string(index=False))